# The P gate on motifbreakR with five models: Malinois and Kircher

The gate that worked with NT-v2 500M (`ΔP Gate on motifbreakR`, `ΔP Gate 2`): for every motifbreakR call,
alleleDiff × (percentile of the occurrence's P(bound | ref) among the calls of the same TF); per variant the call with
the largest |gated score|. Nothing fitted. Here the same gate with the function CAVs of all five models of the pooled
run (same ChIP sites, same estimator, one layer each, chosen by held-out ChIP AUC, not by any MPRA):

| model | layer |
|---|---|
| NT-v2 500M | 24 |
| DNABERT-2 | 4 |
| GENA-LM | 11 |
| HyenaDNA | 3 |
| NT 2.5B | 14 |

**Section 1 (GPU, one session per model; NT-v2 500M is already done).** P(bound | ref) of every Malinois motif
occurrence with the model's direction (all-sites fit `w_all`, as for NT-v2 500M), one file per TF in
`malinois/p_<model>/`, resumable. Set `MODEL` in 1.1. Rough times: DNABERT-2 and HyenaDNA well under an hour,
GENA-LM about as long as NT-v2 500M (~45 min), NT 2.5B several hours (run it last, or skip it).

**Section 2 (CPU).** Kircher: P(bound | ref) of every Kircher occurrence is already stored in each model's fit
(out-of-fold, `prob_hit`), so all five models run without a GPU. Malinois: the models whose Section 1 is done.
Per model and TF tier by that model's own CAV AUC: motifbreakR, the gate's gain, null N2 (P shuffled among the calls of
the same TF, 200 times), and for Malinois the AUROC gain for clear effects.

In [1]:
!pip install -q -U "transformers==4.57.6" "tokenizers==0.22.0" "huggingface_hub==0.34.0" "safetensors==0.4.5"
!pip install -q scikit-learn scipy pandas pyfaidx
from google.colab import drive
drive.mount("/content/drive")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.34.0 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.34.0 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.34.0 which is incompatible.
diffusers 0.42.1 requires safetensors>=0.8.0, but you have safetensors 0.4.5 which is incompatible.
Mounted at /content/drive


---
# Section 1 — P(bound | ref) of every Malinois occurrence, one model *(GPU runtime)*

## 1.1 Settings (set `MODEL`); readout copied unchanged from `Pooled Regions 1c`

In [2]:
MODEL = "hyenadna"          # "dnabert2", "genalm", "hyenadna" or "nt2p5b"; one GPU session per model

WORK_DIR = "/content/drive/MyDrive/GitHub/Biological-Foundation-Model/Notebooks/EP Pair Evaluation/per_occurrence/motif_token_study"
import os, json, time, gc, sys, shutil
import numpy as np, pandas as pd
import torch
from pyfaidx import Fasta
from transformers import AutoTokenizer, AutoModel, AutoConfig, AutoModelForMaskedLM, AutoModelForCausalLM

C = json.load(open(f"{WORK_DIR}/config.json"))
POOL = f"{WORK_DIR}/fcav_pool"
MAL = f"{WORK_DIR}/malinois"
MWJ = json.load(open(f"{POOL}/mpra_windows.json"))
REGISTRY = {
    "nt500m":   dict(hf="InstaDeepAI/nucleotide-transformer-v2-500m-multi-species", tok_map=("kmer", 6), causal=False, batch=32),
    "dnabert2": dict(hf="zhihan1996/DNABERT-2-117M",                                tok_map=("offsets",), causal=False, batch=32, out="dnabert2"),
    "genalm":   dict(hf="AIRI-Institute/gena-lm-bert-large-t2t",                    tok_map=("offsets",), causal=False, batch=32),
    "hyenadna": dict(hf="LongSafari/hyenadna-large-1m-seqlen-hf",                   tok_map=("char",), causal=True, batch=32, rc_mode="concat"),
    "nt2p5b":   dict(hf="InstaDeepAI/nucleotide-transformer-2.5b-multi-species",    tok_map=("kmer", 6), causal=False, batch=16),
}
LAYER_OF = {"nt500m": 24, "dnabert2": 4, "genalm": 11, "hyenadna": 3, "nt2p5b": 14}
LAYER = LAYER_OF[MODEL]
FIT_LAYERS = {MODEL: [LAYER]}            # only the layer used is kept by the readout
BASE_WEIGHTED = False
BATCH = 32
WLEN, MOFF = C["window_len"], C["motif_offset"]
DEV = torch.device("cuda")
COMP = str.maketrans("ACGTN", "TGCAN")
rc = lambda s: s.translate(COMP)[::-1]
FITD = f"{POOL}/fits/{MODEL}"
OUTP = f"{MAL}/p_{MODEL}"
os.makedirs(OUTP, exist_ok=True)
LOCAL_FA = "/content/hg38.fa"
if not os.path.exists(LOCAL_FA):
    print("copying hg38 to local disk (~3 GB, one-off)...")
    shutil.copy(C["genome_fa"], LOCAL_FA)
    if os.path.exists(C["genome_fa"] + ".fai"):
        shutil.copy(C["genome_fa"] + ".fai", LOCAL_FA + ".fai")
GENOME = Fasta(LOCAL_FA, sequence_always_upper=True)
H = pd.read_csv(f"{MAL}/hits.csv")
U = H[~H["leak"]].drop_duplicates("uid")
print(f"{MODEL}, layer {LAYER}: {len(U):,} occurrences, {U['motif'].nunique()} TFs")

copying hg38 to local disk (~3 GB, one-off)...
hyenadna, layer 3: 344,201 occurrences, 251 TFs


In [3]:
def load_tokenizer(tag):
    spec = REGISTRY[tag]
    tokz = AutoTokenizer.from_pretrained(spec["hf"], trust_remote_code=True)
    tokz.padding_side = "right"          # several of these default to left (HyenaDNA)
    if tokz.pad_token is None:
        tokz.pad_token = tokz.eos_token if tokz.eos_token is not None else tokz.unk_token
    return tokz


def token_bases(spec, n):
    # (n, 2) base interval of each sequence token, for the arithmetic maps
    kind = spec["tok_map"][0]
    if kind == "kmer":
        k = spec["tok_map"][1]
        return np.stack([np.arange(n) * k, (np.arange(n) + 1) * k], 1)
    if kind == "char":
        return np.stack([np.arange(n), np.arange(n) + 1], 1)
    raise RuntimeError("offsets are resolved by the tokenizer, not here")


def encode(tokz, spec, seqs):
    # input_ids (B, T), attention mask (B, T), and per sequence: the positions of its sequence tokens in input_ids
    # and the base interval each one covers
    kind = spec["tok_map"][0]
    if kind == "offsets":
        assert tokz.is_fast, f"{spec['hf']} has no fast tokenizer, so offset mapping is unavailable"
        enc = tokz(seqs, return_tensors="pt", padding=True, return_offsets_mapping=True)
    else:
        enc = tokz(seqs, return_tensors="pt", padding=True)
    ids = enc["input_ids"]
    am = enc.get("attention_mask")
    if am is None:
        am = (ids != tokz.pad_token_id).long()
    amn = am.numpy()
    pos, spans = [], []
    if kind == "offsets":
        om = enc["offset_mapping"].numpy()
        for b in range(len(seqs)):
            keep = (om[b, :, 1] > om[b, :, 0]) & (amn[b] == 1)
            pos.append(np.flatnonzero(keep))
            spans.append(om[b, keep])
    else:
        for b in range(len(seqs)):
            sp = np.array(tokz.get_special_tokens_mask(ids[b].tolist(), already_has_special_tokens=True))
            p = np.flatnonzero((sp == 0) & (amn[b] == 1))
            pos.append(p)
            spans.append(token_bases(spec, len(p)))
        assert len({len(p) for p in pos}) == 1, f"{spec['hf']}: windows of equal length gave different token counts"
    return ids, am, pos, spans


def span_weights(bases, lo, hi):
    # per-token weight for the base interval [lo, hi)
    ov = np.minimum(bases[:, 1], hi) - np.maximum(bases[:, 0], lo)
    ov = np.clip(ov, 0, None).astype(np.float32)
    if not BASE_WEIGHTED:
        ov = (ov > 0).astype(np.float32)
    assert ov.sum() > 0, f"no token covers bases [{lo}, {hi})"
    return ov / ov.sum()


def load_model(tag):
    spec = REGISTRY[tag]
    # the class the checkpoint ships its own code for (NT-v2 has AutoModelForMaskedLM but no AutoModel)
    amap = getattr(AutoConfig.from_pretrained(spec["hf"], trust_remote_code=True), "auto_map", None) or {}
    cls = next((c for k, c in (("AutoModel", AutoModel), ("AutoModelForMaskedLM", AutoModelForMaskedLM),
                               ("AutoModelForCausalLM", AutoModelForCausalLM)) if k in amap), AutoModel)
    mdl = cls.from_pretrained(spec["hf"], trust_remote_code=True).to(DEV).eval()
    # DNABERT-2's triton flash-attention kernel does not compile on current triton; its code falls back to PyTorch
    for name in {type(m).__module__ for m in mdl.modules()}:
        mod = sys.modules.get(name)
        if mod is not None and getattr(mod, "flash_attn_qkvpacked_func", None) is not None:
            mod.flash_attn_qkvpacked_func = None
            print(f"{tag}: triton flash-attention disabled in {name.split('.')[-1]}; using PyTorch attention")
    if spec.get("out") == "dnabert2" and getattr(mdl, "pooler", None) is not None:
        mdl.pooler = None
    return mdl


POS, DEPTH, LOADED = {}, {}, {}


@torch.no_grad()
def hidden_stack(tag, model, ids, am):
    # one (layers, T_j, H) float tensor per sequence; only the layers in POS[tag] once prepare() has set it
    spec = REGISTRY[tag]
    if spec.get("out") == "dnabert2":
        # DNABERT-2 returns every encoder layer unpadded; one sequence at a time keeps the token order
        out = []
        for j in range(ids.shape[0]):
            t = ids[j][am[j].bool()][None].to(DEV)
            layers, _ = model(t, attention_mask=torch.ones_like(t), output_all_encoded_layers=True)
            S = torch.stack([x.reshape(t.shape[1], -1) for x in layers], 0).float()
            out.append(S[POS[tag]] if tag in POS else S)
        return out
    kw = {} if spec["causal"] else {"attention_mask": am.to(DEV)}
    hs = model(input_ids=ids.to(DEV), output_hidden_states=True, **kw).hidden_states
    if tag in POS:
        hs = [hs[i] for i in POS[tag]]
    S = torch.stack(hs, 1).float()
    return [S[j] for j in range(S.shape[0])]


PROBE = [h["window"] for h in MWJ["hits"][:4]]


def prepare(tag):
    # load the model; from one forward pass: number of layers, token tiling, and where FIT_LAYERS sit in the stack
    spec = REGISTRY[tag]
    mdl = load_model(tag)
    tokz = LOADED.get(tag, {}).get("tokz") or load_tokenizer(tag)
    LOADED[tag] = dict(spec=spec, tokz=tokz)
    POS.pop(tag, None)
    ids, am, pos, spans = encode(tokz, spec, PROBE[:2])
    S = hidden_stack(tag, mdl, ids, am)
    for j, Sj in enumerate(S):
        assert Sj.ndim == 3 and int(pos[j].max()) < Sj.shape[1]
        b = spans[j]
        assert b[0, 0] == 0 and b[-1, 1] == WLEN and (b[1:, 0] == b[:-1, 1]).all(), f"{tag}: tokens do not tile"
    nl = S[0].shape[0]
    lids = list(range(1, nl + 1)) if spec.get("out") == "dnabert2" else list(range(nl))   # DNABERT-2: no embedding layer
    DEPTH[tag] = lids[-1]
    miss = [l for l in FIT_LAYERS[tag] if l not in lids]
    assert not miss, f"{tag}: layers {miss} do not exist (ids {lids[0]}..{lids[-1]})"
    POS[tag] = [lids.index(l) for l in FIT_LAYERS[tag]]
    print(f"{tag}: {nl} layers (depth {DEPTH[tag]}), fitting {FIT_LAYERS[tag]}, hidden {S[0].shape[2]}, "
          f"{len(pos[0])} tokens for {WLEN} bp")
    del S
    return mdl


@torch.no_grad()
def readout(tag, model, seqs, lohi):
    # (n, layers, D): D = hidden, or 2 * hidden for a causal model (forward ‖ reverse complement)
    L = LOADED[tag]
    tokz, spec = L["tokz"], L["spec"]
    passes = [(seqs, lohi)]
    if spec["causal"]:
        passes.append(([rc(s) for s in seqs], [(WLEN - b, WLEN - a) for a, b in lohi]))
    bs = spec.get("batch", BATCH)
    parts = []
    for ss, lh in passes:
        out = None
        for i in range(0, len(ss), bs):
            chunk, lhc = ss[i:i + bs], lh[i:i + bs]
            ids, am, pos, spans = encode(tokz, spec, chunk)
            S = hidden_stack(tag, model, ids, am)
            if out is None:
                out = np.zeros((len(ss), S[0].shape[0], S[0].shape[2]), np.float16)
            for j in range(len(chunk)):
                w = torch.tensor(span_weights(spans[j], *lhc[j]), device=DEV)
                p = torch.tensor(pos[j], device=DEV)
                out[i + j] = (S[j][:, p] * w[None, :, None]).sum(1).half().cpu().numpy()
            del S
        parts.append(out)
    if len(parts) == 1:
        return parts[0]
    if spec.get("rc_mode", "concat") == "mean":
        return ((parts[0].astype(np.float32) + parts[1].astype(np.float32)) / 2).astype(np.float16)
    return np.concatenate(parts, 2)

## 1.2 Every occurrence (one file per TF; an interrupted session continues where it stopped)

Check first: the same formula on the Kircher hits of 20 TFs, against the out-of-fold P(bound) stored in the fit
(they rank the hits almost the same for NT-v2 500M: Spearman 0.994).

In [4]:
from scipy import stats


def direction(m):
    z = np.load(f"{FITD}/{m}.npz", allow_pickle=False)
    li = [int(l) for l in z["layers"]].index(LAYER)
    return z, li, z["w_all"][li].astype(np.float64), z["mu"][li].astype(np.float64), z["sd"][li].astype(np.float64)


def p_bound(m, windows, w_motif, mdl):
    _, _, w, mu, sd = direction(m)
    X = readout(MODEL, mdl, windows, [(MOFF, MOFF + w_motif)] * len(windows))[:, 0].astype(np.float64)
    return 1 / (1 + np.exp(-(((X - mu) / sd) @ w[:-1] + w[-1])))


mdl = prepare(MODEL)
HBY = {}
for h in MWJ["hits"]:
    HBY.setdefault(h["motif"], []).append(h)
chk = []
for m in sorted(HBY, key=lambda m: -len(HBY[m]))[:20]:
    z, li, *_ = direction(m)
    ref = pd.Series(z["prob_hit"][:, li], index=pd.MultiIndex.from_arrays([z["hit_region"], z["hit_id"]]))
    hs = HBY[m]
    p = p_bound(m, [h["window"] for h in hs], hs[0]["end"] - hs[0]["start"], mdl)
    chk.append(np.c_[p, ref.loc[[(h["region"], h["hit_id"]) for h in hs]].values])
chk = np.vstack(chk)
print(f"check, {len(chk)} Kircher hits of 20 TFs: Spearman(all-sites P, out-of-fold P) = {stats.spearmanr(chk[:, 0], chk[:, 1])[0]:.3f}")

todo = [(m, g) for m, g in U.groupby("motif") if not os.path.exists(f"{OUTP}/{m}.csv")]
n_left = sum(len(g) for _, g in todo)
print(f"{len(todo)} TFs, {n_left:,} occurrences to do")
t_all, done = time.time(), 0
for k, (m, g) in enumerate(todo):
    w = int(g["b"].iloc[0] - g["a"].iloc[0])
    wins = [GENOME[c][s - MOFF:s - MOFF + WLEN].seq for c, s in zip(g["chrom38"], g["g38"])]
    assert all(len(x) == WLEN and "N" not in x for x in wins)
    p = p_bound(m, wins, w, mdl)
    pd.DataFrame({"uid": g["uid"].values, "P": p}).to_csv(f"{OUTP}/{m}.part.csv", index=False)
    os.replace(f"{OUTP}/{m}.part.csv", f"{OUTP}/{m}.csv")
    done += len(g)
    el = time.time() - t_all
    if k % 10 == 0 or k == len(todo) - 1:
        print(f"[{k + 1}/{len(todo)}] {m:10s} {len(g):6,} occurrences  [{el / 60:.1f} min; ~{el / done * (n_left - done) / 60:.0f} min left]", flush=True)
del mdl; gc.collect(); torch.cuda.empty_cache()

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/982 [00:00<?, ?B/s]

configuration_hyena.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/LongSafari/hyenadna-large-1m-seqlen-hf:
- configuration_hyena.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modeling_hyena.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/LongSafari/hyenadna-large-1m-seqlen-hf:
- modeling_hyena.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


... (2 outputs left out)

tokenization_hyena.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/LongSafari/hyenadna-large-1m-seqlen-hf:
- tokenization_hyena.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


special_tokens_map.json:   0%|          | 0.00/971 [00:00<?, ?B/s]

hyenadna: 10 layers (depth 9), fitting [3], hidden 256, 600 tokens for 600 bp
check, 438 Kircher hits of 20 TFs: Spearman(all-sites P, out-of-fold P) = 0.996
251 TFs, 344,201 occurrences to do
[1/251] ARNT2         828 occurrences  [0.0 min; ~10 min left]
[11/251] CGGBP1      1,119 occurrences  [0.2 min; ~10 min left]
[21/251] E2F8        1,227 occurrences  [0.5 min; ~10 min left]
[31/251] ETS1          406 occurrences  [0.8 min; ~9 min left]
[41/251] FOXK1         242 occurrences  [1.0 min; ~9 min left]
[51/251] GABPA       1,502 occurrences  [1.2 min; ~9 min left]
[61/251] HNF4A         776 occurrences  [1.3 min; ~9 min left]
[71/251] KLF13       1,484 occurrences  [1.7 min; ~8 min left]
[81/251] MEF2A       1,195 occurrences  [2.2 min; ~8 min left]
[91/251] NACC2       5,482 occurrences  [2.6 min; ~7 min left]
[101/251] NR2F2         367 occurrences  [2.9 min; ~7 min left]
[111/251] ONECUT1       391 occurrences  [3.2 min; ~6 min left]
[121/251] RARA        2,104 occurrences  [3.6 m

In [5]:
from google.colab import runtime
runtime.unassign()

---
# Section 2 — the gate, five models *(CPU runtime)*

## 2.1 Shared

In [1]:
from google.colab import drive
drive.mount('/content/drive')
WORK_DIR = "/content/drive/MyDrive/GitHub/Biological-Foundation-Model/Notebooks/EP Pair Evaluation/per_occurrence/motif_token_study"
import os, json, re, time
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score

MAL = f"{WORK_DIR}/malinois"
POOL = f"{WORK_DIR}/fcav_pool"
OUT = f"{WORK_DIR}/gate_five_models"
os.makedirs(OUT, exist_ok=True)
MODELS = ["nt500m", "dnabert2", "genalm", "hyenadna", "nt2p5b"]
LAYER_OF = {"nt500m": 24, "dnabert2": 4, "genalm": 11, "hyenadna": 3, "nt2p5b": 14}
N_PERM = 200
pd.set_option("display.width", 250)
fmt = lambda x: f"{x:+.3f}"


def load_fit(model):
    # per TF: held-out ChIP AUC at the model's layer, and the out-of-fold P(bound) of every Kircher hit
    d, L = f"{POOL}/fits/{model}", LAYER_OF[model]
    auc, kp = {}, {}
    for f in os.listdir(d):
        if not f.endswith(".npz") or f.endswith(".part.npz"):
            continue
        z = np.load(f"{d}/{f}", allow_pickle=False)
        li = [int(l) for l in z["layers"]].index(L)
        auc[f[:-4]] = float(np.asarray(z["auc"])[li])
        for r, h, p in zip(z["hit_region"].astype(int), z["hit_id"].astype(int), z["prob_hit"][:, li]):
            kp[r, h] = float(p)
    return auc, kp


FITS = {m: load_fit(m) for m in MODELS}
print(pd.DataFrame({m: pd.Series(a).describe() for m, (a, _) in FITS.items()}).T[["count", "mean", "50%", "min", "max"]]
      .rename(columns={"50%": "median"}).to_string(float_format=lambda x: f"{x:.3f}"))


def pear(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    if a.std() == 0 or b.std() == 0:
        return np.nan
    return float(np.corrcoef(a, b)[0, 1])


def auroc(lab, s):
    lab = np.asarray(lab, bool)
    return roc_auc_score(lab, s) if 0 < lab.sum() < len(lab) else np.nan


def tf_pct(x, g):
    o = np.lexsort((x, g))
    gs = g[o]
    st = np.r_[0, np.flatnonzero(gs[1:] != gs[:-1]) + 1]
    cnt = np.diff(np.r_[st, len(gs)])
    q = np.empty(len(x)); q[o] = (np.arange(len(gs)) - np.repeat(st, cnt) + 0.5) / np.repeat(cnt, cnt)
    return q


def pick(v, grp, n):
    out = np.zeros(n)
    if len(v):
        o = np.lexsort((-np.abs(v), grp))
        f = o[np.r_[True, grp[o][1:] != grp[o][:-1]]]
        out[grp[f]] = v[f]
    return out


def gate(T, p, n):
    return pick(T["ad"] * tf_pct(p, T["tf"]), T["grp"], n)


def shuffle_tf(T, rng):
    o = np.lexsort((rng.random(len(T["tf"])), T["tf"]))
    out = np.empty(len(o)); out[np.argsort(T["tf"], kind="stable")] = T["p"][o]
    return out

Mounted at /content/drive
           count  mean  median   min   max
nt500m   251.000 0.721   0.719 0.467 0.983
dnabert2 251.000 0.748   0.748 0.419 0.984
genalm   251.000 0.750   0.747 0.484 0.981
hyenadna 251.000 0.719   0.714 0.472 0.979
nt2p5b   251.000 0.771   0.771 0.451 0.984


## 2.2 Kircher (all five models, no GPU needed)

motifbreakR calls as in `Pooled Regions 2` (region 0 / 8 / 20 Pearson 0.177 / 0.279 / 0.464 must come back); a call's
P = the largest out-of-fold P(bound) of that TF's occurrences containing the SNV. Mean Pearson over the 4 HepG2/K562
regions and over the 17 others; tiers by each model's own CAV AUC.

In [2]:
HOME = {"HepG2", "K562"}
K_TIERS = [("all TFs", 0.0), ("AUC >= 0.75", 0.75)]
MBR_THRESH, MBR_NEUT = 0.85, 0.4
C = json.load(open(f"{WORK_DIR}/config.json"))
MWJ = json.load(open(f"{POOL}/mpra_windows.json"))
INFO = {int(k): v for k, v in MWJ["regions"].items()}
def _read_meme(path):
    out, name, rows_, ns = {}, None, [], 20
    for line in open(path):
        t = line.split()
        if line.startswith("MOTIF"):
            if name is not None and name not in out:
                out[name] = (np.array(rows_), ns)
            name, rows_, ns = (t[2] if len(t) > 2 else t[1]), [], 20
        elif line.startswith("letter-probability"):
            mm = re.search(r"nsites=\s*([0-9.]+)", line)
            ns = float(mm.group(1)) if mm else 20
        elif name is not None and len(t) == 4 and all(re.fullmatch(r"[0-9.eE+-]+", x) for x in t):
            rows_.append([float(x) for x in t])
    if name is not None and name not in out:
        out[name] = (np.array(rows_), ns)
    return out


MBR_MATS = {}
for name, (f, ns) in _read_meme(C["meme_path"]).items():
    f = f / f.sum(1, keepdims=True)
    pp = (f * ns + 0.25) / (ns + 1)
    S = pp * (pp.max(1) - pp.min(1))[:, None]
    S = np.vstack([S.T, np.zeros(len(S))]).T
    MBR_MATS[name] = (S, S[::-1][:, [3, 2, 1, 0, 4]], S[:, :4].min(1).sum(), S[:, :4].max(1).sum())
ENC = {"A": 0, "C": 1, "G": 2, "T": 3}
WIN = {}
for h in MWJ["hits"]:
    WIN.setdefault(h["region"], []).append(h)


def build(r, HS):
    # verbatim from Pooled Regions 2: SNV x hit rows and motifbreakR calls on the hits HS of region r
    S_ = f"{POOL}/regions/r{r}"
    hits = pd.read_csv(f"{S_}/hits.csv").set_index("hit_id")
    snv = pd.read_csv(f"{S_}/snvs.csv")
    q = pd.read_csv(f"{S_}/snv_hit_dpwm.csv")
    q = q[q["hit_id"].isin(HS)].sort_values(["snv", "hit_id"]).reset_index(drop=True)
    sn = q["snv"].values
    st = np.r_[0, np.flatnonzero(sn[1:] != sn[:-1]) + 1]
    hi = np.searchsorted(HS, q["hit_id"].values)
    n_snv = len(st)
    MOT = hits.loc[HS, "motif"].values
    base = {}
    for h in WIN[r]:
        for i, b in enumerate(h["window"]):
            base.setdefault(h["start"] - C["motif_offset"] + i, b)
    lo = min(base)
    seq = np.array([ENC.get(base.get(x, "N"), 4) for x in range(lo, max(base) + 1)])
    s_ = snv.loc[sn[st]]
    pos, ref, alt = s_["local"].values - lo, s_["ref"].map(ENC).values, s_["alt"].map(ENC).values
    assert (seq[pos] == ref).all(), "rebuilt sequence disagrees with the SNV reference bases"
    out = []
    for m in sorted(set(MOT)):
        if m not in MBR_MATS:
            continue
        S, Sr, smin, smax = MBR_MATS[m]
        w = len(S)
        n = len(seq) - w + 1
        idx = np.arange(n)[:, None] + np.arange(w)[None, :]
        f_fwd, f_rev = S[np.arange(w), seq[idx]].sum(1), Sr[np.arange(w), seq[idx]].sum(1)
        stw = pos[:, None] - np.arange(w)[None, :]
        k = np.broadcast_to(np.arange(w), stw.shape)
        ok = (stw >= 0) & (stw < n)
        stc = np.clip(stw, 0, n - 1)
        ref_f, ref_r = np.where(ok, f_fwd[stc], -np.inf), np.where(ok, f_rev[stc], -np.inf)
        alt_f = ref_f + np.where(ok, S[k, alt[:, None]] - S[k, ref[:, None]], 0)
        alt_r = ref_r + np.where(ok, Sr[k, alt[:, None]] - Sr[k, ref[:, None]], 0)
        sr, sa = np.maximum(ref_f, ref_r).max(1), np.maximum(alt_f, alt_r).max(1)
        pr, pa = (sr - smin) / (smax - smin), (sa - smin) / (smax - smin)
        diff = sa - sr
        keep = np.isfinite(sr) & ((pr > MBR_THRESH) | (pa > MBR_THRESH)) & (np.abs(diff) >= MBR_NEUT)
        out += [(i, m, diff[i]) for i in np.flatnonzero(keep)]
    t = pd.DataFrame(out, columns=["i", "motif", "alleleDiff"])
    t["j"] = np.arange(len(t))
    rows_ = pd.DataFrame({"i": np.repeat(np.arange(n_snv), np.diff(np.r_[st, len(sn)])), "motif": MOT[hi], "h": hi})
    link = t[["j", "i", "motif"]].merge(rows_, on=["i", "motif"])
    keep = np.sort(link["j"].unique())
    t = t.loc[keep].reset_index(drop=True)
    return dict(snv=snv, n_snv=n_snv, snv_id=sn[st], y=snv.loc[sn[st], "log2FC"].values,
                DIFF=t["alleleDiff"].values, TI=t["i"].values, MOTP=t["motif"].values)


SH = {}                                     # (region, snv) -> list of (motif, hit_id) containing it
for r in sorted(INFO):
    q = pd.read_csv(f"{POOL}/regions/r{r}/snv_hit_dpwm.csv")
    hm = pd.read_csv(f"{POOL}/regions/r{r}/hits.csv").set_index("hit_id")["motif"]
    q["motif"] = hm.reindex(q["hit_id"]).values
    SH[r] = q


def kircher_model(model, thr):
    auc, kp = FITS[model]
    regs, rows = {}, []
    for r in sorted(INFO):
        hs = np.array(sorted(h["hit_id"] for h in WIN.get(r, []) if h["motif"] in auc and auc[h["motif"]] >= thr))
        if not len(hs):
            continue
        R = build(r, hs)
        if R["n_snv"] < 20:
            continue
        regs[r] = R
        q = SH[r][SH[r]["hit_id"].isin(hs)].copy()
        q["P"] = [kp.get((r, h), np.nan) for h in q["hit_id"]]
        pm = q.groupby(["snv", "motif"])["P"].max()
        snv = R["snv_id"][R["TI"]]
        rows.append(pd.DataFrame({"region": r, "i": R["TI"], "motif": R["MOTP"], "ad": R["DIFF"],
                                  "p": pm.reindex(pd.MultiIndex.from_arrays([snv, R["MOTP"]])).values}))
    Cc = pd.concat(rows, ignore_index=True)
    Cc = Cc[np.isfinite(Cc["p"])].reset_index(drop=True)
    off = dict(zip(regs, np.cumsum([0] + [regs[r]["n_snv"] for r in regs])[:-1]))
    T = dict(ad=Cc["ad"].values, tf=pd.factorize(Cc["motif"])[0], grp=(Cc["region"].map(off) + Cc["i"]).values, p=Cc["p"].values)
    return regs, off, T


def k_scores(regs, off, T, p):
    n = sum(R["n_snv"] for R in regs.values())
    mb, gt = pick(T["ad"], T["grp"], n), gate(T, p, n)
    return {r: (pear(mb[off[r]:off[r] + R["n_snv"]], R["y"]), pear(gt[off[r]:off[r] + R["n_snv"]], R["y"])) for r, R in regs.items()}


KROWS, KREG = [], []
for model in MODELS:
    for tier, thr in K_TIERS:
        regs, off, T = kircher_model(model, thr)
        obs = k_scores(regs, off, T, T["p"])
        if model == MODELS[0] and tier == "all TFs":
            print("check (motifbreakR): " + ", ".join(f"region {r} {obs[r][0]:.3f} (was {w})" for r, w in ((0, 0.177), (8, 0.279), (20, 0.464)) if r in obs))
        rng = np.random.default_rng(7)
        null = [k_scores(regs, off, T, shuffle_tf(T, rng)) for _ in range(N_PERM)]
        for r, (a, b) in obs.items():
            KREG.append({"model": model, "tier": tier, "region": r, "cell": INFO[r]["cell"], "motifbreakR": a, "gate": b})
        for h in (True, False):
            rs = [r for r in regs if (set(INFO[r]["cell"].split(",")) <= HOME) == h]
            mb, gt = np.array([obs[r][0] for r in rs]), np.array([obs[r][1] for r in rs])
            ng = np.array([np.nanmean([n_[r][1] for r in rs]) for n_ in null])
            KROWS.append({"model": model, "tier": tier, "regions": f"{'HepG2/K562' if h else 'other cells'} ({len(rs)})",
                          "TFs": int(len(np.unique(T["tf"]))), "motifbreakR": np.nanmean(mb), "gate": np.nanmean(gt),
                          "gain": np.nanmean(gt) - np.nanmean(mb), "wins": f"{int((gt > mb).sum())}/{len(rs)}",
                          "p N2": np.mean(ng >= np.nanmean(gt))})
    print(f"{model} done", flush=True)
KS = pd.DataFrame(KROWS).set_index(["tier", "regions", "model"]).sort_index(level=[0, 1], sort_remaining=False)
pd.DataFrame(KREG).to_csv(f"{OUT}/kircher_regions.csv", index=False); KS.to_csv(f"{OUT}/kircher_summary.csv")
print("\nKircher: mean Pearson over regions, motifbreakR vs the P gate of each model\n")
print(KS.to_string(float_format=fmt))

check (motifbreakR): region 0 0.177 (was 0.177), region 8 0.279 (was 0.279), region 20 0.464 (was 0.464)
nt500m done
dnabert2 done
genalm done
hyenadna done
nt2p5b done

Kircher: mean Pearson over regions, motifbreakR vs the P gate of each model

                                       TFs  motifbreakR   gate   gain   wins   p N2
tier        regions          model                                                 
AUC >= 0.75 HepG2/K562 (4)   nt500m     87       +0.377 +0.475 +0.098    4/4 +0.000
                             dnabert2  115       +0.347 +0.402 +0.055    3/4 +0.010
                             genalm    112       +0.370 +0.430 +0.060    3/4 +0.005
                             hyenadna   87       +0.381 +0.451 +0.070    3/4 +0.000
                             nt2p5b    143       +0.347 +0.382 +0.035    3/4 +0.255
            other cells (17) nt500m     87       +0.066 +0.092 +0.026  13/17 +0.000
                             dnabert2  115       +0.103 +0.121 +0.018  13/17 +0.0

## 2.3 Malinois (the models whose Section 1 is done)

Variants with a motifbreakR call on a TF with a P (as in the NT-v2 500M runs). Pearson with Δ, gain of the gate over
motifbreakR, p N2, and the AUROC gain for clear effects (|Δ| ≥ 2 SE). Tiers by each model's own CAV AUC.

In [3]:
CELLS3 = ["HepG2", "K562", "SKNSH"]
TIERS = [("all TFs", 0.0), ("AUC >= 0.70", 0.70), ("AUC >= 0.75", 0.75), ("AUC >= 0.80", 0.80)]
ACTIVE_MIN, Z_CLEAR = 1.0, 2.0
V = pd.read_csv(f"{MAL}/variants.csv.gz", index_col="vi", dtype={"chrom": str}, low_memory=False)
H = pd.read_csv(f"{MAL}/hits.csv")
H = H[~H["leak"]][["vi", "uid", "motif"]]
CALLS = pd.read_csv(f"{MAL}/calls.csv")
rd = lambda d: pd.concat([pd.read_csv(f"{d}/{f}") for f in os.listdir(d) if f.endswith(".csv") and not f.endswith(".part.csv")])
done = [m for m in MODELS if os.path.isdir(f"{MAL}/p_{m}") and len([f for f in os.listdir(f"{MAL}/p_{m}") if f.endswith(".csv") and not f.endswith(".part.csv")]) >= 240]
print("Malinois P(bound) available for:", done)

MROWS = []
for model in done:
    auc, _ = FITS[model]
    PU = rd(f"{MAL}/p_{model}").set_index("uid")["P"]
    hp = H.assign(P=PU.reindex(H["uid"]).values).dropna(subset=["P"])
    CM = CALLS.merge(hp.groupby(["vi", "motif"])["P"].max().reset_index(), on=["vi", "motif"])
    CM["auc"] = CM["motif"].map(auc)
    for tier, thr in TIERS:
        L = CM[CM["auc"] >= thr].sort_values(["vi", "motif"]).reset_index(drop=True)
        VI = np.unique(L["vi"].values)
        T = dict(ad=L["alleleDiff"].values, tf=pd.factorize(L["motif"])[0], grp=np.searchsorted(VI, L["vi"].values), p=L["P"].values)
        n = len(VI)
        Y = {c: V.loc[VI, f"d_{c}"].values for c in CELLS3}
        CLR = {c: np.abs(Y[c]) / V.loc[VI, f"se_{c}"].values >= Z_CLEAR for c in CELLS3}
        SETS = {(c, s): np.isfinite(Y[c]) & ((V.loc[VI, f"act_{c}"].values >= ACTIVE_MIN) if s == "active" else True)
                for c in CELLS3 for s in ("all", "active")}
        mb = pick(T["ad"], T["grp"], n)
        gt = gate(T, T["p"], n)
        rng = np.random.default_rng(7)
        nulls = [gate(T, shuffle_tf(T, rng), n) for _ in range(N_PERM)]
        for (c, s), m in SETS.items():
            y = Y[c][m]
            r_mb, r_gt = pear(mb[m], y), pear(gt[m], y)
            MROWS.append({"model": model, "tier": tier, "cell": c, "variants": s, "TFs": int(L["motif"].nunique()), "n": int(m.sum()),
                          "motifbreakR": r_mb, "gain": r_gt - r_mb,
                          "p N2": np.mean([pear(g_[m], y) >= r_gt for g_ in nulls]),
                          "AUROC gain": auroc(CLR[c][m], np.abs(gt[m])) - auroc(CLR[c][m], np.abs(mb[m]))})
    print(f"{model} done", flush=True)
MS = pd.DataFrame(MROWS).set_index(["tier", "cell", "variants", "model"]).sort_index(level=[0, 1, 2], sort_remaining=False)
MS.to_csv(f"{OUT}/malinois.csv")
print("\nMalinois: Pearson gain of the P gate over motifbreakR, per model\n")
print(MS.to_string(float_format=fmt))
print("\ncompact: Pearson gain (active variants)\n")
print(MS.xs("active", level="variants")["gain"].unstack("model")[done].to_string(float_format=fmt))

Malinois P(bound) available for: ['nt500m', 'dnabert2', 'genalm', 'hyenadna']
nt500m done
dnabert2 done
genalm done
hyenadna done

Malinois: Pearson gain of the P gate over motifbreakR, per model

                                     TFs      n  motifbreakR   gain   p N2  AUROC gain
tier        cell  variants model                                                      
AUC >= 0.70 HepG2 active   nt500m    143  12078       +0.158 +0.021 +0.000      +0.032
                           dnabert2  176  13654       +0.157 +0.017 +0.025      +0.029
                           genalm    175  13525       +0.156 +0.013 +0.125      +0.027
                           hyenadna  142  11862       +0.155 +0.024 +0.000      +0.042
                  all      nt500m    143  48086       +0.085 +0.016 +0.000      +0.031
                           dnabert2  176  54763       +0.081 +0.017 +0.000      +0.033
                           genalm    175  54093       +0.080 +0.015 +0.000      +0.032
                    

## End

Outputs in `gate_five_models/`: `kircher_regions.csv`, `kircher_summary.csv`, `malinois.csv`; Malinois P(bound) per
model in `malinois/p_<model>/`.